NaN analizė

In [1]:
import pandas

from config import features, unscaled_csv, pollutants

unscaled_df = pandas.read_csv(unscaled_csv, parse_dates=['date'])

total = len(unscaled_df)
nan_rows = unscaled_df[features].isnull().any(axis=1).sum()
nan_cells = unscaled_df[features].isnull().sum().sum()
print(f"Eilutės su bent vienu NaN: {nan_rows} iš {total} ({nan_rows / total * 100:.1f}%)")
print(f"Iš viso NaN reikšmių: {nan_cells}")
print(f"Vidutiniškai NaN stulpelių per NaN eilutę: {nan_cells / nan_rows:.1f}")

Eilutės su bent vienu NaN: 30770 iš 122736 (25.1%)
Iš viso NaN reikšmių: 57069
Vidutiniškai NaN stulpelių per NaN eilutę: 1.9


NaN per stulpelį

In [2]:
print(unscaled_df.isnull().sum())


date                0
day_category        0
time                0
O3               4359
NO               3653
NO2              3653
CO              12704
PM10            10379
PM2.5            9649
wind_speed       3168
temp             3168
wind_dir_sin     3168
wind_dir_cos     3168
dtype: int64


Teršalų koreliacija

In [3]:
print(unscaled_df[pollutants].corr()['CO'].round(2))

CO       1.00
O3      -0.49
NO       0.72
NO2      0.68
PM10     0.50
PM2.5    0.49
Name: CO, dtype: float64


Tarpų analizė

In [4]:
for col in features:
    is_null = unscaled_df[col].isnull()
    groups = is_null.ne(is_null.shift()).cumsum()
    gap_sizes = unscaled_df[is_null].groupby(groups).size()

    print(f"\n=== {col} — tarpų: {len(gap_sizes)}, iš viso NaN: {is_null.sum()}, ilgiausias: {gap_sizes.max() if len(gap_sizes) > 0 else 0} val. ===")
    print(f"  1 val.:    {(gap_sizes == 1).sum()}")
    print(f"  2-5 val.:  {((gap_sizes >= 2) & (gap_sizes <= 5)).sum()}")
    print(f"  6-24 val.: {((gap_sizes >= 6) & (gap_sizes <= 24)).sum()}")
    print(f"  25+ val.:  {(gap_sizes > 24).sum()}")


=== CO — tarpų: 601, iš viso NaN: 12704, ilgiausias: 1551 val. ===
  1 val.:    417
  2-5 val.:  92
  6-24 val.: 40
  25+ val.:  52

=== O3 — tarpų: 614, iš viso NaN: 4359, ilgiausias: 405 val. ===
  1 val.:    346
  2-5 val.:  152
  6-24 val.: 83
  25+ val.:  33

=== NO — tarpų: 529, iš viso NaN: 3653, ilgiausias: 314 val. ===
  1 val.:    321
  2-5 val.:  134
  6-24 val.: 37
  25+ val.:  37

=== NO2 — tarpų: 529, iš viso NaN: 3653, ilgiausias: 314 val. ===
  1 val.:    321
  2-5 val.:  134
  6-24 val.: 37
  25+ val.:  37

=== PM10 — tarpų: 467, iš viso NaN: 10379, ilgiausias: 1967 val. ===
  1 val.:    213
  2-5 val.:  133
  6-24 val.: 38
  25+ val.:  83

=== PM2.5 — tarpų: 545, iš viso NaN: 9649, ilgiausias: 1656 val. ===
  1 val.:    256
  2-5 val.:  161
  6-24 val.: 49
  25+ val.:  79

=== wind_speed — tarpų: 98, iš viso NaN: 3168, ilgiausias: 144 val. ===
  1 val.:    0
  2-5 val.:  0
  6-24 val.: 83
  25+ val.:  15

=== temp — tarpų: 98, iš viso NaN: 3168, ilgiausias: 144 val. 

Tarpai per metus — gal kurie metai ypač prasti?

In [5]:
unscaled_df['year'] = unscaled_df['date'].dt.year
print(unscaled_df.groupby('year')[['CO', 'PM10', 'PM2.5', 'O3']].apply(lambda x: x.isnull().sum()))

        CO  PM10  PM2.5   O3
year                        
2012   415  1296    694  344
2013    80   491    276  142
2014   253   605    275  402
2015   221   258    262  159
2016  1251   404    349  246
2017  2431   315    331  155
2018   208   300    913  129
2019   286   314    812  186
2020  1888  2210   1873  702
2021  1426  2120   1265  637
2022  1938   398   1290   92
2023   543   507    520  450
2024  1384   713    473  179
2025   380   448    316  536


Išmetus CO

In [6]:
features_no_co = [f for f in features if f != 'CO']


nan_without_co = unscaled_df[features_no_co].isnull().any(axis=1).sum()
print(f"Be CO:  {nan_without_co} iš {len(unscaled_df)} ({nan_without_co / len(unscaled_df) * 100:.1f}%)")

Be CO:  21636 iš 122736 (17.6%)


Švarūs 5 valandų blokai per dienos valandas ir kategorijas

In [7]:
for start_hour in range(1, 21):
    target_hours = list(range(start_hour, start_hour + 5))
    for cat in [0, 1, 2]:
        cat_name = ['darbo', 'išeig', 'priešš'][cat]
        subset = unscaled_df[
            (unscaled_df['day_category'] == cat) &
            (unscaled_df['time'].isin(target_hours))
            ]
        clean = subset[features_no_co].notna().all(axis=1)
        clean_days = clean.groupby(subset['date']).apply(lambda g: (len(g) == 5) and g.all()).sum()
        if cat == 0:
            print(f"Val. {start_hour:2d}-{start_hour+4:2d} | ", end="")
        print(f"{cat_name}: {clean_days:4d}", end=" | ")
    print()

Val.  1- 5 | darbo: 2246 | išeig: 1259 | priešš:  599 | 
Val.  2- 6 | darbo: 2277 | išeig: 1279 | priešš:  606 | 
Val.  3- 7 | darbo: 2280 | išeig: 1293 | priešš:  607 | 
Val.  4- 8 | darbo: 2269 | išeig: 1285 | priešš:  606 | 
Val.  5- 9 | darbo: 2227 | išeig: 1280 | priešš:  603 | 
Val.  6-10 | darbo: 2162 | išeig: 1274 | priešš:  597 | 
Val.  7-11 | darbo: 2097 | išeig: 1273 | priešš:  591 | 
Val.  8-12 | darbo: 2027 | išeig: 1269 | priešš:  590 | 
Val.  9-13 | darbo: 1950 | išeig: 1272 | priešš:  591 | 
Val. 10-14 | darbo: 1920 | išeig: 1273 | priešš:  587 | 
Val. 11-15 | darbo: 1908 | išeig: 1283 | priešš:  590 | 
Val. 12-16 | darbo: 1944 | išeig: 1281 | priešš:  598 | 
Val. 13-17 | darbo: 2013 | išeig: 1279 | priešš:  605 | 
Val. 14-18 | darbo: 2089 | išeig: 1278 | priešš:  608 | 
Val. 15-19 | darbo: 2162 | išeig: 1274 | priešš:  613 | 
Val. 16-20 | darbo: 2217 | išeig: 1278 | priešš:  620 | 
Val. 17-21 | darbo: 2261 | išeig: 1285 | priešš:  624 | 
Val. 18-22 | darbo: 2281 | išei

Visų features koreliacija

In [8]:
print(unscaled_df[features].corr().round(2))

                CO    O3    NO   NO2  PM10  PM2.5  wind_speed  temp  \
CO            1.00 -0.49  0.72  0.68  0.50   0.49       -0.09 -0.06   
O3           -0.49  1.00 -0.56 -0.56 -0.36  -0.37        0.21  0.20   
NO            0.72 -0.56  1.00  0.87  0.53   0.48       -0.01 -0.01   
NO2           0.68 -0.56  0.87  1.00  0.52   0.48       -0.03  0.09   
PM10          0.50 -0.36  0.53  0.52  1.00   0.87       -0.19 -0.09   
PM2.5         0.49 -0.37  0.48  0.48  0.87   1.00       -0.26 -0.12   
wind_speed   -0.09  0.21 -0.01 -0.03 -0.19  -0.26        1.00  0.17   
temp         -0.06  0.20 -0.01  0.09 -0.09  -0.12        0.17  1.00   
wind_dir_sin -0.17  0.25 -0.22 -0.17  0.11   0.17       -0.19 -0.06   
wind_dir_cos -0.23  0.27 -0.29 -0.32 -0.18  -0.10       -0.12 -0.16   

              wind_dir_sin  wind_dir_cos  
CO                   -0.17         -0.23  
O3                    0.25          0.27  
NO                   -0.22         -0.29  
NO2                  -0.17         -0.32  
PM1

Duomenų statistika

In [9]:
print(unscaled_df[features].describe().round(2))


              CO         O3         NO        NO2       PM10      PM2.5  \
count  110032.00  118377.00  119083.00  119083.00  112357.00  113087.00   
mean        0.40      22.72      85.34      65.93      22.90      14.47   
std         0.25      18.40      97.84      40.17      13.41      10.61   
min         0.00      -0.93      -0.37       0.00      -2.90      -5.00   
25%         0.23       7.53      17.46      35.57      13.53       7.40   
50%         0.35      18.11      47.77      57.95      20.29      11.80   
75%         0.52      33.88     116.11      88.02      28.90      18.20   
max         2.89     156.46     872.83     321.91     187.90     127.60   

       wind_speed       temp  wind_dir_sin  wind_dir_cos  
count   119568.00  119568.00     119568.00     119568.00  
mean         3.46      10.37         -0.24         -0.04  
std          1.66       6.13          0.72          0.65  
min          0.00     -10.50         -1.00         -1.00  
25%          2.30       6.10 

Sezoniniai teršalų trendai pagal mėnesį

In [10]:
unscaled_df['month'] = unscaled_df['date'].dt.month
print(unscaled_df.groupby('month')[['O3', 'NO', 'NO2', 'PM10', 'PM2.5']].mean().round(2))

          O3      NO    NO2   PM10  PM2.5
month                                    
1      15.82  108.35  70.77  25.35  16.23
2      19.97   96.44  69.10  25.75  16.12
3      25.85   80.21  68.79  27.86  18.62
4      34.58   67.69  65.29  24.27  15.59
5      34.50   64.23  62.18  21.43  13.68
6      29.32   72.99  64.67  19.54  12.27
7      24.30   74.79  64.86  18.40  11.87
8      23.17   69.61  60.90  20.55  12.07
9      19.59   81.90  64.56  21.66  13.84
10     15.14   95.07  66.21  22.73  13.64
11     13.90  106.25  66.16  23.94  15.08
12     16.71  106.22  67.81  22.46  14.55
